In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Configuration
MAX_RUNTIME_HOURS = 11.5
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600
EPOCHS = 20  # 20 Epochs on FULL data is enough
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")
print(f"Maximum runtime: {MAX_RUNTIME_HOURS} hours")

Path("results_final").mkdir(exist_ok=True)

# ==================== DATASET ====================
def get_data_loaders():
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(), # Added Augmentation
        transforms.RandomCrop(32, padding=4), # Added Augmentation
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    
    # Num_workers=2 is safer for Kaggle
    trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return trainloader, testloader

# ==================== VISION TRANSFORMER (Unchanged) ====================
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=192):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class MLP(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, use_norm=True, use_res=True):
        super().__init__()
        self.use_norm = use_norm
        self.use_res = use_res
        self.norm1 = nn.LayerNorm(dim) if use_norm else nn.Identity()
        self.attn = MultiHeadAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim) if use_norm else nn.Identity()
        self.mlp = MLP(dim, int(dim * mlp_ratio))
        
    def forward(self, x):
        if self.use_res: x = x + self.attn(self.norm1(x))
        else: x = self.attn(self.norm1(x))
        
        if self.use_res: x = x + self.mlp(self.norm2(x))
        else: x = self.mlp(self.norm2(x))
        return x

class VisionTransformer(nn.Module):
    def __init__(self, img_size=32, patch_size=4, num_classes=10, embed_dim=192, 
                 depth=12, num_heads=3, mlp_ratio=4.0, use_norm=True, use_res=True):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, (img_size//patch_size)**2 + 1, embed_dim))
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads, mlp_ratio, use_norm, use_res) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim) if use_norm else nn.Identity()
        self.head = nn.Linear(embed_dim, num_classes)
        self._init_weights()
        
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed
        for block in self.blocks: x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])

# ==================== UPDATED METRICS LOGGER ====================
class MetricsLogger:
    def __init__(self, model, model_name):
        self.model = model
        self.model_name = model_name
        self.metrics = []
        self.hooks = []
        self.activations = {}
        # Hook only the DEEPEST layer for spectrum analysis (Layer 11)
        self.last_layer_idx = len(model.blocks) - 1
        
        # Register hooks for all layers for Rank
        for idx, block in enumerate(model.blocks):
            hook = block.mlp.register_forward_hook(self._make_hook(f'mlp_{idx}'))
            self.hooks.append(hook)
            
    def _make_hook(self, name):
        def hook(module, input, output):
            self.activations[name] = output.detach()
        return hook
    
    def log_metrics(self, epoch, train_loss, val_loss, val_acc):
        metrics_dict = {
            'model': self.model_name,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_acc': val_acc
        }
        
        # Compute Rank for ALL layers
        for name, activation in self.activations.items():
            # Flatten: (B, N, D) -> (B*N, D)
            x = activation.reshape(-1, activation.shape[-1])
            if x.shape[0] > 2000: x = x[:2000] # Subsample for speed
            
            # SVD
            try:
                s = torch.linalg.svdvals(x.float())
                s = s[s > 1e-6] # Numerical threshold
                if len(s) > 0:
                    stable_rank = (s**2).sum() / (s[0]**2)
                    metrics_dict[f'rank_{name}'] = stable_rank.item()
                    
                    # Store Spectrum ONLY for the last layer and final epoch
                    if 'mlp_11' in name and epoch == EPOCHS:
                        # Save top 20 singular values normalized
                        spectrum = (s[:20] / s[0]).cpu().numpy()
                        metrics_dict['spectrum'] = spectrum
                else:
                    metrics_dict[f'rank_{name}'] = 1.0
            except:
                metrics_dict[f'rank_{name}'] = 1.0

        self.metrics.append(metrics_dict)
        self.activations.clear()
        return metrics_dict
    
    def get_dataframe(self):
        # Flatten the spectrum array if it exists, otherwise just return metrics
        data = []
        for m in self.metrics:
            d = m.copy()
            if 'spectrum' in d:
                del d['spectrum'] # Don't put array in CSV cell
            data.append(d)
        return pd.DataFrame(data)

    def get_final_spectrum(self):
        # Helper to retrieve the spectrum from the last epoch
        for m in reversed(self.metrics):
            if 'spectrum' in m:
                return m['spectrum']
        return None

    def cleanup(self):
        for hook in self.hooks: hook.remove()

# ==================== UPDATED PLOTTING ====================
def save_plots(all_metrics, all_spectra):
    print("\n📊 Generating Enhanced Plots...")
    
    # Plot 1: Rank Heatmaps
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    for idx, (model_name, df) in enumerate(all_metrics.items()):
        if len(df) == 0: continue
        rank_cols = [c for c in df.columns if 'rank_mlp_' in c]
        # Sort columns naturally
        rank_cols.sort(key=lambda x: int(x.split('_')[-1]))
        
        rank_data = df[rank_cols].values.T
        sns.heatmap(rank_data, ax=axes[idx], cmap='viridis', vmin=1.0, vmax=10.0) # Fixed scale
        axes[idx].set_title(f'{model_name}')
        axes[idx].set_ylabel('Layer')
        axes[idx].set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig('results_final/heatmap.png')
    plt.close()

    # Plot 2: THE SPECTRAL SCREE PLOT (New!)
    fig, ax = plt.subplots(figsize=(10, 6))
    for model_name, spectrum in all_spectra.items():
        if spectrum is not None:
            ax.plot(range(1, len(spectrum)+1), spectrum, marker='o', label=model_name, linewidth=2)
    
    ax.set_yscale('log')
    ax.set_xlabel('Singular Value Index')
    ax.set_ylabel('Normalized Singular Value (Log Scale)')
    ax.set_title('Spectral Collapse: Deepest Layer Spectrum')
    ax.grid(True, which="both", ls="-", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig('results_final/spectral_scree.png')
    plt.close()

    # Plot 3: Validation Accuracy
    fig, ax = plt.subplots(figsize=(10, 6))
    for model_name, df in all_metrics.items():
        ax.plot(df['epoch'], df['val_acc'], marker='o', label=model_name)
    ax.set_title('Validation Accuracy')
    ax.set_ylabel('Accuracy (%)')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True)
    plt.savefig('results_final/accuracy.png')
    plt.close()

# ==================== MAIN ====================
def run_experiment():
    start_total = time.time()
    trainloader, testloader = get_data_loaders()
    
    configs = {
        'ViT_Star': {'use_norm': True, 'use_res': True},
        'ViT_NoNorm': {'use_norm': False, 'use_res': True},
        'ViT_NoRes': {'use_norm': True, 'use_res': False},
        'ViT_Naked': {'use_norm': False, 'use_res': False},
    }
    
    all_metrics = {}
    all_spectra = {}
    
    for model_name, cfg in configs.items():
        print(f"\n🚀 Training {model_name}...")
        try:
            model = VisionTransformer(depth=12, **cfg).to(DEVICE)
            optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            criterion = nn.CrossEntropyLoss()
            logger = MetricsLogger(model, model_name)
            
            for epoch in range(1, EPOCHS + 1):
                # Check runtime
                if time.time() - start_total > MAX_RUNTIME_SECONDS: break
                
                # TRAIN LOOP (FIXED: Runs on FULL dataset)
                model.train()
                t_loss, count = 0, 0
                for data, target in trainloader:
                    data, target = data.to(DEVICE), target.to(DEVICE)
                    optimizer.zero_grad()
                    output = model(data)
                    loss = criterion(output, target)
                    loss.backward()
                    optimizer.step()
                    t_loss += loss.item()
                    count += 1
                
                # VALIDATION
                model.eval()
                correct, total = 0, 0
                with torch.no_grad():
                    for data, target in testloader:
                        data, target = data.to(DEVICE), target.to(DEVICE)
                        output = model(data)
                        _, pred = output.max(1)
                        total += target.size(0)
                        correct += pred.eq(target).sum().item()
                
                # LOGGING (Uses subset for speed)
                logger.log_metrics(epoch, t_loss/count, 0.0, 100.*correct/total)
                print(f"  Ep {epoch}: Acc {100.*correct/total:.2f}%")
            
            all_metrics[model_name] = logger.get_dataframe()
            all_spectra[model_name] = logger.get_final_spectrum()
            logger.cleanup()
            del model, optimizer
            torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"Error {model_name}: {e}")
            
    save_plots(all_metrics, all_spectra)
    print("DONE. Download results_final folder.")

if __name__ == "__main__":
    run_experiment()

Using device: cuda
Maximum runtime: 11.5 hours


100%|██████████| 170M/170M [00:10<00:00, 15.8MB/s]



🚀 Training ViT_Star...
  Ep 1: Acc 38.82%
  Ep 2: Acc 43.76%
  Ep 3: Acc 48.41%
  Ep 4: Acc 49.74%
  Ep 5: Acc 52.25%
  Ep 6: Acc 52.56%
  Ep 7: Acc 54.29%
  Ep 8: Acc 54.64%
  Ep 9: Acc 53.79%
  Ep 10: Acc 55.91%
  Ep 11: Acc 58.39%
  Ep 12: Acc 57.32%
  Ep 13: Acc 59.41%
  Ep 14: Acc 59.89%
  Ep 15: Acc 59.08%
  Ep 16: Acc 59.44%
  Ep 17: Acc 61.23%
  Ep 18: Acc 61.56%
  Ep 19: Acc 62.00%
  Ep 20: Acc 62.99%

🚀 Training ViT_NoNorm...
  Ep 1: Acc 30.83%
  Ep 2: Acc 36.17%
  Ep 3: Acc 40.61%
  Ep 4: Acc 47.50%
  Ep 5: Acc 50.63%
  Ep 6: Acc 53.29%
  Ep 7: Acc 55.95%
  Ep 8: Acc 58.96%
  Ep 9: Acc 60.69%
  Ep 10: Acc 60.44%
  Ep 11: Acc 64.24%
  Ep 12: Acc 64.15%
  Ep 13: Acc 65.47%
  Ep 14: Acc 64.95%
  Ep 15: Acc 63.63%
  Ep 16: Acc 66.60%
  Ep 17: Acc 67.11%
  Ep 18: Acc 64.62%
  Ep 19: Acc 67.83%
  Ep 20: Acc 68.70%

🚀 Training ViT_NoRes...
  Ep 1: Acc 10.00%
  Ep 2: Acc 10.00%
  Ep 3: Acc 10.00%
  Ep 4: Acc 10.00%
  Ep 5: Acc 10.00%
  Ep 6: Acc 10.00%
  Ep 7: Acc 10.00%
  Ep 8: Ac